# Tale Role — serve adapters from Colab

Loads **our** private Hub LoRA. Closing Colab does not change Hub weights.

1. Runtime → GPU (T4).
2. Upload `services/llm-runner/serve.py` from this repo (same folder as the notebook files).
3. Colab → Secrets → `HF_TOKEN` (read on the private repos). Never put the token in a cell.

One T4 = one role. This notebook is storyteller. Mechanics = new Colab + `levonov/talerole-mechanics`.

In [ ]:
!pip install -q "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" huggingface_hub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
from pathlib import Path
assert Path("serve.py").exists(), "Upload services/llm-runner/serve.py into this Colab first"

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_LOAD_IN_4BIT"] = "1"
os.environ["HF_MODEL_ID"] = "levonov/talerole-storyteller"
print("ready", os.environ["HF_MODEL_ID"])

In [ ]:
import subprocess, time, re, sys, os

server = subprocess.Popen(
    [sys.executable, "serve.py", "--role", "storyteller", "--hf-model", os.environ["HF_MODEL_ID"], "--port", "8091"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for _ in range(180):
    line = server.stdout.readline()
    if line:
        print(line, end="")
        if "llm-runner" in line:
            break
    if server.poll() is not None:
        print(server.stdout.read())
        raise SystemExit("runner exited")
    time.sleep(1)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8091"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    if not line:
        continue
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("\nPUBLIC URL:", url)
print("Render: LLM_STORYTELLER_URL=" + (url or ""))